# 04 — Comparison and pre-registered decisions

Paired match bootstrap (1000 draws, seed 20261108, 95 %). With seven matches the intervals are wide: read decisions, not third decimals.

In [1]:
from pathlib import Path
import os, sys, json, time
ROOT = Path.cwd()
if not (ROOT / 'exp2_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_2')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import numpy as np
import pandas as pd
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
import exp2_common as C
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])
import exp2_protocol as P, exp2_report as R
import exp1_common as C1
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PREREG verified 154a10f7cda6821e | frozen 2026-11-08T13:06:55+00:00


In [2]:
parts = [pd.read_csv(C.RESULTS / 'reference_rows.csv'), pd.read_csv(C.RESULTS / 'argmax_rows.csv')]
for variant in P.VARIANTS:
    for view in C.VIEWS:
        parts.append(pd.DataFrame(C1.read_json(C.RESULTS / f'variant_{variant}__{view}.json')['rows']))
rows = pd.concat(parts, ignore_index=True)
R.assert_proxy_only(rows)
rows.to_csv(C.RESULTS / 'per_match_results.csv', index=False)
table = R.summary_table(rows)
table.to_csv(C.RESULTS / 'comparison_table.csv', index=False)
cols = ['variant', 'mean_spell_f1', 'mean_macro_f1', 'mean_abs_count_error', 'max_abs_count_error', 'delta_vs_exp1_ref', 'ci95_low', 'ci95_high', 'wins_vs_exp1_ref', 'count_gate']
for view in C.VIEWS:
    print(f'== {view}'); print(R.as_markdown(table[table.view == view], cols))

== original
| variant | mean_spell_f1 | mean_macro_f1 | mean_abs_count_error | max_abs_count_error | delta_vs_exp1_ref | ci95_low | ci95_high | wins_vs_exp1_ref | count_gate |
|---|---|---|---|---|---|---|---|---|---|
| exp1_ref | 0.656 | 0.918 | 0.212 | 0.342 | +0.000 | +0.000 | +0.000 | 0 | False |
| exp2_aug | 0.657 | 0.917 | 0.200 | 0.354 | +0.001 | -0.008 | +0.010 | 3 | False |
| exp2_aug_const | 0.657 | 0.917 | 0.200 | 0.354 | +0.001 | -0.008 | +0.010 | 3 | False |
| exp2_aug_hsmm | 0.656 | 0.917 | 0.190 | 0.337 | +0.001 | -0.009 | +0.010 | 4 | False |
| exp2_aug_s1 | 0.651 | 0.918 | 0.192 | 0.344 | -0.005 | -0.016 | +0.006 | 3 | False |
| exp2_aug_s2 | 0.656 | 0.917 | 0.220 | 0.335 | +0.001 | -0.008 | +0.011 | 3 | False |
| exp2_aug_ens3 | 0.666 | 0.922 | 0.204 | 0.360 | +0.011 | +0.004 | +0.019 | 6 | False |
| exp1_ref_argmax | 0.624 | 0.918 | 0.089 | 0.182 | -0.032 | -0.043 | -0.022 | 0 | True |
| exp2_aug_argmax | 0.628 | 0.917 | 0.082 | 0.209 | -0.027 | -0.043 | -0.014 | 0 |

## Robustness: target_like minus original, per arm (paired by match)

In [3]:
shift = pd.DataFrame(R.view_shift(rows))
print(shift.round(4).to_string(index=False))

        variant  mean_change                                            ci95
       exp1_ref      -0.0077  [-0.011762513534384289, -0.003780784038296018]
       exp2_aug      -0.0000   [-0.005151387245286553, 0.004772920274217818]
 exp2_aug_const      -0.0000   [-0.005151387245286553, 0.004772920274217818]
  exp2_aug_hsmm      -0.0018 [-0.0053453021993390085, 0.0020972799642246524]
    exp2_aug_s1      -0.0018 [-0.0053923201151169315, 0.0021204372339513095]
    exp2_aug_s2      -0.0007  [-0.006789647624522414, 0.0034879822772453566]
  exp2_aug_ens3      -0.0020  [-0.005315940982379911, 0.0012906102121789278]
exp1_ref_argmax      -0.0081 [-0.011142466677478246, -0.0049879687046177675]
exp2_aug_argmax      -0.0016  [-0.004959851011945677, 0.0014246233475370632]
      ryu_final      -0.0087 [-0.014034294184758633, -0.0038090337665458995]
      rasya_c08      -0.0591    [-0.07284755164306513, -0.04521653065632048]
      rasya_c06      -0.0624   [-0.07191586371299515, -0.053303266933327424]

## Quality gates (exp_0) per view

In [4]:
gate_rows = [R.gates(rows, view, v) for view in C.VIEWS for v in R.LABELS
             if v in set(rows[rows.view == view].variant) and v != 'baseline']
print(pd.DataFrame([{'view': g['view'], 'variant': g['variant'], **g['checks'], 'pass': g['pass'],
                     'worst_count_error': round(g['worst_count_error'], 3), 'worst_match': g['worst_count_match']}
                    for g in gate_rows]).to_string(index=False))

       view         variant  gain_at_least_005  ci_lower_above_zero  macro_regression_at_most_001  all_match_counts_within_30pct  pass  worst_count_error worst_match
   original        exp1_ref               True                 True                          True                          False False              0.342      J03WMX
   original        exp2_aug               True                 True                          True                          False False              0.354      J03WMX
   original  exp2_aug_const               True                 True                          True                          False False              0.354      J03WMX
   original   exp2_aug_hsmm               True                 True                          True                          False False              0.337      J03WMX
   original     exp2_aug_s1               True                 True                          True                          False False              0.344      J03WMX
   o

## Pre-registered decisions (PREREG §6)

In [5]:
decision = R.decide(rows)
C1.write_json(C.RESULTS / 'decision.json', decision)
report = {'prereg_sha256': protocol['prereg_sha256'], 'bootstrap': 'paired match bootstrap, 1000 draws, seed 20261108, 95% percentile',
          'table': table.to_dict(orient='records'), 'deltas_vs_exp1_ref': R.deltas_vs_reference(rows),
          'view_shift': R.view_shift(rows), 'gates': gate_rows, 'decision': decision,
          'reproduction_exp1_ref': C1.read_json(C.RESULTS / 'reproduction_exp1_ref.json')}
C1.write_json(C.RESULTS / 'proxy_report.json', report)
for name, rule in decision['rules'].items():
    shown = {k: rule[k] for k in ('mean_delta', 'ci95', 'wins', 'seed_spread', 'max_abs_count_error') if k in rule}
    print(f"{name:34s} {'PASSED' if rule['passed'] else 'FAILED':7s} {rule['rule']:45s} {shown}")
print('supported:', decision['supported'], '| adopt exp_2 candidate:', decision['adopt_exp2_candidate'])

H1a_transfer_target_like           PASSED  CI lower > 0                                  {'mean_delta': 0.008737955802174053, 'ci95': [0.0009657088171498958, 0.018155064939567783], 'wins': 5}
H1b_in_domain_original             PASSED  mean delta >= -0.02                           {'mean_delta': 0.0010702821835275125, 'ci95': [-0.007634491692396308, 0.010193334912714902], 'wins': 3}
H2_count_gate_target_like          FAILED  all 7 matches within +-30%                    {'max_abs_count_error': 0.3405088062622309}
H2_non_inferiority_target_like     PASSED  CI lower > -0.01 vs constant decoder          {'mean_delta': -0.0022456008997761012, 'ci95': [-0.006367605235168161, 0.001412647339812819], 'wins': 4}
H3_ensemble_target_like            PASSED  |mean delta| > seed spread and CI excludes 0  {'mean_delta': 0.007490034193133274, 'ci95': [0.001366980199571331, 0.013770939238786023], 'wins': 5, 'seed_spread': 0.00748288412439746}
supported: {'H1': True, 'H2': False, 'H3': True} | adopt exp_

## Figure (proxy only)

In [6]:
order = ['baseline', 'rasya_c06', 'rasya_c08', 'ryu_final', 'exp1_ref', 'exp2_aug', 'exp2_aug_const', 'exp2_aug_hsmm', 'exp2_aug_ens3']
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for view, marker in zip(C.VIEWS, ('o', 's')):
    part = table[table.view == view].set_index('variant').reindex(order)
    axes[0].plot(order, part.mean_spell_f1, marker=marker, label=view)
    axes[1].plot(order, part.max_abs_count_error, marker=marker, label=view)
axes[0].set_ylabel('mean spellF1 (7 held-out matches)'); axes[0].set_title('spellF1 by arm and view')
axes[1].axhline(0.3, color='0.5', ls='--', label='±30% gate'); axes[1].set_ylabel('max |count error|')
axes[1].set_title('worst-match count error')
for ax in axes:
    ax.tick_params(axis='x', rotation=35); ax.legend()
fig.tight_layout(); fig.savefig(C.RESULTS / 'fig_exp2_views.png', dpi=130)
print('saved', C.RESULTS / 'fig_exp2_views.png')

saved /home/jeremy-mboe/Documents/Kuliah/Sem5/barca/exp_jeremy/exp_2/results/fig_exp2_views.png
